# BIOMASS L1 Orthorectification — GCP-Based Geolocation with Delaunay Interpolation

**Objective**: Orthorectify ESA BIOMASS L1 SCS (slant-range complex) SAR data using Ground Control Point (GCP) geolocation.

## Overview

**BIOMASS L1 geolocation** differs from SICD:
- No analytical R/Rdot model
- Instead: **Sparse GCP grid** (lat/lon/height at specific row/col)
- Geolocation uses **Delaunay triangulation** to interpolate between GCPs

This notebook demonstrates:
1. Loading BIOMASS L1 SCS products
2. GCP-based geolocation (`GCPGeolocation`)
3. Pauli decomposition in slant range
4. Orthorectification to UTM grid

## Theory

### GCP Grid Interpolation

BIOMASS provides GCPs at regular intervals (e.g., every 100 pixels):
$$\text{GCP}_{i,j} = (row_i, col_j, lat, lon, h)$$

**Delaunay triangulation** constructs triangles connecting GCPs. For any pixel $(r, c)$:
1. Find the triangle containing $(r, c)$
2. Compute barycentric coordinates $(\alpha, \beta, \gamma)$
3. Interpolate: $lat = \alpha \cdot lat_1 + \beta \cdot lat_2 + \gamma \cdot lat_3$

### Quad-Pol Pauli Decomposition

BIOMASS is **quad-pol** (HH, HV, VH, VV). Pauli RGB:
- **R** (red): $|HH - VV|$ (double-bounce)
- **G** (green): $|HV + VH|$ (volume scattering)
- **B** (blue): $|HH + VV|$ (surface scattering)

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | BIOMASS reader via factory (`get_reader('biomass', ...)`) |
| `grdl.geolocation` | `create_geolocation()` auto-detects GCP geolocation |
| `grdl.image_processing` | `PauliDecomposition` |
| `grdl.image_processing.ortho` | `orthorectify()`, `UTMGrid` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.decomposition import PauliDecomposition
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Configuration — User Paths

In [ ]:
# Path to BIOMASS L1 SCS product directory
BIOMASS_PATH = Path("/path/to/your/BIO_S1_RAW__1S_20230101T000000_20230101T000100")

# Validate path
assert BIOMASS_PATH.exists(), f"BIOMASS product not found: {BIOMASS_PATH}"

print(f"✓ BIOMASS product: {BIOMASS_PATH.name}")

## Step 1: Load BIOMASS Metadata and Quad-Pol Chip

In [ ]:
with get_reader('biomass', BIOMASS_PATH) as reader:
    meta = reader.metadata
    rows, cols = meta.shape
    
    print(f"Image size: {rows} × {cols}")
    print(f"Polarizations: {meta.polarizations}")
    print(f"GCP grid: {meta.gcp_grid_shape} (row × col samples)")
    
    # Extract center chip (4096×4096 or smaller)
    chip_size = min(4096, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    # Read quad-pol cube
    hh = reader.read(pol='HH', row_start=r_start, row_end=r_start+chip_size,
                     col_start=c_start, col_end=c_start+chip_size)
    hv = reader.read(pol='HV', row_start=r_start, row_end=r_start+chip_size,
                     col_start=c_start, col_end=c_start+chip_size)
    vh = reader.read(pol='VH', row_start=r_start, row_end=r_start+chip_size,
                     col_start=c_start, col_end=c_start+chip_size)
    vv = reader.read(pol='VV', row_start=r_start, row_end=r_start+chip_size,
                     col_start=c_start, col_end=c_start+chip_size)

print(f"\nExtracted chip: {hh.shape} per pol")

## Step 2: Pauli Decomposition in Slant Range

In [ ]:
# Stack quad-pol cube
quad_pol = np.stack([hh, hv, vh, vv], axis=0)  # (4, rows, cols)

# Apply Pauli decomposition
pauli = PauliDecomposition()
pauli_rgb = pauli.to_rgb(quad_pol)  # (rows, cols, 3)

print(f"Pauli RGB: {pauli_rgb.shape}")

## Step 3: Create GCP Geolocation

In [ ]:
with get_reader('biomass', BIOMASS_PATH) as reader:
    # Auto-detect GCP geolocation
    geo_full = create_geolocation(reader)
    
    # Wrap with chip offset
    geo = ChipGeolocation(
        geolocation=geo_full,
        row_offset=r_start,
        col_offset=c_start,
    )

print(f"✓ Geolocation: {type(geo_full).__name__}")
print(f"  GCP count: {len(geo_full.gcps) if hasattr(geo_full, 'gcps') else 'N/A'}")

## Step 4: Define UTM Output Grid

In [ ]:
# Use center pixel to determine UTM zone
center_lat, center_lon, _ = geo.image_to_latlon(chip_size//2, chip_size//2)

output_grid = UTMGrid.from_center(
    center_lat=center_lat,
    center_lon=center_lon,
    width_m=chip_size * 5.0,  # BIOMASS ~5m pixel spacing
    height_m=chip_size * 5.0,
    pixel_size_m=5.0,
)

print(f"UTM Grid: Zone {output_grid.utm_zone}{output_grid.hemisphere}")
print(f"  Size: {output_grid.shape}")

## Step 5: Orthorectify Pauli RGB

In [ ]:
print("Orthorectifying Pauli RGB (3 bands)...")

# Orthorectify each RGB channel separately
ortho_r = orthorectify(pauli_rgb[:,:,0], geo, output_grid, interpolation='bilinear').data
ortho_g = orthorectify(pauli_rgb[:,:,1], geo, output_grid, interpolation='bilinear').data
ortho_b = orthorectify(pauli_rgb[:,:,2], geo, output_grid, interpolation='bilinear').data

ortho_pauli_rgb = np.stack([ortho_r, ortho_g, ortho_b], axis=-1)

print(f"✓ Orthorectified Pauli RGB: {ortho_pauli_rgb.shape}")

## Step 6: Visualization — Slant vs Ground Pauli

In [ ]:
viewer = napari.Viewer(title="BIOMASS Ortho + Pauli")

# Layer 1: Slant-range Pauli RGB
viewer.add_image(
    pauli_rgb,
    name="Pauli Slant Range",
    rgb=True,
)

# Layer 2: Orthorectified Pauli RGB
viewer.add_image(
    ortho_pauli_rgb,
    name="Pauli UTM Ortho",
    rgb=True,
    visible=True,
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer opened")
print("  Layer 1: Pauli RGB in slant-range geometry")
print("  Layer 2: Orthorectified Pauli RGB on UTM grid")
print("\nExecution paused — close the napari window to continue and free memory...")
napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del hh, hv, vh, vv, quad_pol, pauli_rgb, ortho_r, ortho_g, ortho_b, ortho_pauli_rgb, geo, geo_full, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Physical Interpretation

### GCP vs R/Rdot Geolocation

| Method | Accuracy | Speed | Use Case |
|--------|----------|-------|----------|
| **R/Rdot** (SICD) | Exact (cm-level) | Slow (iterative) | High-precision SAR |
| **GCP** (BIOMASS) | Interpolated (m-level) | Fast (triangulation) | GCP-based products |

**GCP limitations**:
- Accuracy degrades between GCP grid points
- Assumes linear interpolation (terrain curvature not modeled)
- No DEM integration in Delaunay interpolation

### Pauli Color Interpretation

In orthorectified Pauli RGB:
- **Blue** (surface): Calm water, roads, bare soil
- **Red** (double-bounce): Buildings, tree trunks + ground
- **Green** (volume): Forest canopy, vegetation
- **Yellow** (R+G): Urban areas (mixed double-bounce + volume)
- **Magenta** (R+B): Oriented urban structures
- **Cyan** (G+B): Wetlands, flooded vegetation

## Summary

Successfully demonstrated:
- ✅ BIOMASS L1 quad-pol loading via IO factory
- ✅ GCP-based geolocation (`GCPGeolocation` via `create_geolocation()`)
- ✅ Pauli decomposition in slant range (before ortho)
- ✅ Multi-band orthorectification (R, G, B channels separately)
- ✅ Side-by-side slant vs ground Pauli visualization

### Key GRDL Patterns
- `get_reader('biomass', path)` for BIOMASS products
- `create_geolocation(reader)` auto-detects GCP geolocation
- Process multi-pol data in slant range before ortho
- Orthorectify each channel separately for RGB output

### Next Steps
- Compare GCP vs R/Rdot accuracy (overlay on reference imagery)
- Orthorectify full scene (not just chip)
- Export to GeoTIFF: `GeoTIFFWriter.write(ortho_pauli_rgb, ...)`
- Validate GCP grid spacing vs ortho errors